# Model Inference — Best Model Selection, Model Registry, Final Predictions

Compares internal WMAE across all 8 architectures, selects the best, registers it to the Model Registry (correct current API), loads it back from there, generates final predictions.

## Section 1 — Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import wandb
import pickle
import os

from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_columns', 100)
np.random.seed(42)

ENTITY = "gormo22-free-university-of-tbilisi-"
PROJECT = "walmart-sales-forecasting"
print(f"Using: {ENTITY}/{PROJECT}")

Mounted at /content/drive
Using: gormo22-free-university-of-tbilisi-/walmart-sales-forecasting


## Section 2 — Compare Internal WMAE, Select the Best Model

Your 4 numbers are final, confirmed, tuned. Replace those 4 lines once he sends real numbers.

In [2]:
results = pd.DataFrame([
    {"Model": "LightGBM", "Owner": "Sandro",    "Internal_WMAE": 1571.02, "Method": "20-trial CV-averaged search, clean walk-forward folds"},
    {"Model": "Prophet",  "Owner": "Sandro",    "Internal_WMAE": 2110.64, "Method": "15-trial search (changepoint/seasonality/holidays prior scale), full-scale 2965 series"},
    {"Model": "N-BEATS",  "Owner": "Sandro",    "Internal_WMAE": 3091.24, "Method": "9-trial search (lr/trend_degree/harmonics), holiday-aware, 4-fold CV"},
    {"Model": "DLinear",  "Owner": "Sandro",    "Internal_WMAE": 3955.55, "Method": "9-trial search (lr/kernel_size), holiday-aware, full-scale CV"},
    {"Model": "XGBoost", "Owner": "Gega", "Internal_WMAE": 1442.29, "Method": "4-fold walk-forward CV, 20-trial regularized HPO, k=30 features"},
    {"Model": "TFT",     "Owner": "Gega", "Internal_WMAE": 1417.55, "Method": "Pooled global model, full validation panel (3216 series, 32160 predictions)"},
    {"Model": "SARIMA",  "Owner": "Gega", "Internal_WMAE": 2175.62, "Method": "60-series random-sample pooled validation"},
    {"Model": "ARIMA",   "Owner": "Gega", "Internal_WMAE": 3071.48, "Method": "180-series random-sample pooled validation"},
    {"Model": "TimesFM", "Owner": "Sandro", "Internal_WMAE": 4258.41, "Method": "Zero-shot, pretrained (no training), full-scale 2994 series"},
])

print(results.sort_values("Internal_WMAE").to_string(index=False))

BEST_MODEL = results.sort_values("Internal_WMAE").iloc[0]["Model"]

# Deliberate override: TFT scored marginally lower (1417.55 vs 1442.29), but that gap is
# well within XGBoost's own CV noise (+/-272.09), and XGBoost's pipeline is far simpler to
# deploy (TFT needs a separate historical-data input and time_idx-based output). XGBoost is
# the model actually wired into Sections 4-8 and registered below.
BEST_MODEL = "XGBoost"
print(f"Best model (selected for deployment): {BEST_MODEL}")

placeholders = results[results["Internal_WMAE"] == 9999.99]["Model"].tolist()
if placeholders:
    print(f"Still placeholder, not real yet: {', '.join(placeholders)}")
else:
    print("All results are final.")

   Model  Owner  Internal_WMAE                                                                                 Method
     TFT   Gega        1417.55            Pooled global model, full validation panel (3216 series, 32160 predictions)
 XGBoost   Gega        1442.29                        4-fold walk-forward CV, 20-trial regularized HPO, k=30 features
LightGBM Sandro        1571.02                                  20-trial CV-averaged search, clean walk-forward folds
 Prophet Sandro        2110.64 15-trial search (changepoint/seasonality/holidays prior scale), full-scale 2965 series
  SARIMA   Gega        2175.62                                              60-series random-sample pooled validation
   ARIMA   Gega        3071.48                                             180-series random-sample pooled validation
 N-BEATS Sandro        3091.24                   9-trial search (lr/trend_degree/harmonics), holiday-aware, 4-fold CV
 DLinear Sandro        3955.55                          

## Section 3 — Wandb Login

In [3]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abali22 (gormo22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Section 4 — Reconstruct the Winning Pipeline Class



In [4]:
import xgboost as xgb
import pandas as pd
import numpy as np

class XGBoostPreprocessingPipeline:
    def __init__(self, dept_avg_map, store_avg_map, best_partner, overall_mean_sales,
                 markdown_cols, train_history):
        self.dept_avg_map = dept_avg_map
        self.store_avg_map = store_avg_map
        self.best_partner = best_partner
        self.overall_mean_sales = overall_mean_sales
        self.markdown_cols = markdown_cols
        self.train_history = train_history[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        self.holiday_dates = {
            'Is_SuperBowl':    pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']),
            'Is_LaborDay':     pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']),
            'Is_Thanksgiving': pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']),
            'Is_Christmas':    pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']),
        }

    def transform(self, raw_df, stores_df, features_df):
        data = pd.merge(raw_df, stores_df, on='Store', how='left')
        data = pd.merge(data, features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        data['Date'] = pd.to_datetime(data['Date'])
        has_target = 'Weekly_Sales' in data.columns
        data[self.markdown_cols] = data[self.markdown_cols].fillna(0)
        data = data.sort_values(['Store', 'Date']).reset_index(drop=True)
        data['CPI'] = data.groupby('Store')['CPI'].transform(lambda s: s.ffill().bfill())
        data['Unemployment'] = data.groupby('Store')['Unemployment'].transform(lambda s: s.ffill().bfill())
        data['Year'] = data['Date'].dt.year
        data['Month'] = data['Date'].dt.month
        data['Week'] = data['Date'].dt.isocalendar().week.astype(int)
        data['Day'] = data['Date'].dt.day
        data['Type'] = data['Type'].map({'A': 3, 'B': 2, 'C': 1})
        data['IsHoliday'] = data['IsHoliday'].astype(int)
        data['Total_Markdown'] = data[self.markdown_cols].sum(axis=1)
        data['Has_Markdown'] = (data['Total_Markdown'] > 0).astype(int)
        data['Markdown_Count'] = (data[self.markdown_cols] > 0).sum(axis=1)
        for col, dates in self.holiday_dates.items():
            data[col] = data['Date'].isin(dates).astype(int)
        if not has_target:
            data['Weekly_Sales'] = np.nan
        data['_is_target_row'] = True
        history = self.train_history.copy()
        history['_is_target_row'] = False
        combined = pd.concat([history, data], ignore_index=True, sort=False)
        combined = combined.sort_values(['Store', 'Dept', 'Date', '_is_target_row'])
        combined = combined.drop_duplicates(subset=['Store', 'Dept', 'Date'], keep='last').reset_index(drop=True)
        combined = combined.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
        grp = combined.groupby(['Store', 'Dept'])['Weekly_Sales']
        combined['Sales_Lag1']      = grp.shift(1)
        combined['Sales_Lag2']      = grp.shift(2)
        combined['Sales_Lag3']      = grp.shift(3)
        combined['Sales_Lag52']     = grp.shift(52)
        combined['Sales_RollMean4'] = grp.transform(lambda s: s.shift(1).rolling(4).mean())
        combined['Sales_RollStd4']  = grp.transform(lambda s: s.shift(1).rolling(4).std())
        combined['Partner_Dept'] = combined['Dept'].map(self.best_partner)
        lag1_lookup = combined[['Store', 'Dept', 'Date', 'Sales_Lag1']].rename(
            columns={'Dept': 'Partner_Dept', 'Sales_Lag1': 'Partner_Dept_Sales_Lag1'})
        combined = pd.merge(combined, lag1_lookup, on=['Store', 'Partner_Dept', 'Date'], how='left')
        result = combined[combined['_is_target_row']].drop(columns=['_is_target_row']).reset_index(drop=True)
        if not has_target:
            result = result.drop(columns=['Weekly_Sales'])
        result['Dept_Avg_Sales']  = result['Dept'].map(self.dept_avg_map).fillna(self.overall_mean_sales)
        result['Store_Avg_Sales'] = result['Store'].map(self.store_avg_map).fillna(self.overall_mean_sales)
        return result


class XGBoostFullPipeline:
    def __init__(self, preprocessing_pipeline, model, feature_cols):
        self.preprocessing_pipeline = preprocessing_pipeline
        self.model = model
        self.feature_cols = feature_cols

    def predict(self, raw_df, stores_df, features_df):
        raw_df = raw_df.copy()
        raw_df['Date'] = pd.to_datetime(raw_df['Date'])
        raw_df['Weekly_Sales'] = np.nan

        unique_dates = sorted(raw_df['Date'].unique())
        all_predictions = []

        for current_date in unique_dates:
            rows_so_far = raw_df[raw_df['Date'] <= current_date].copy()
            engineered = self.preprocessing_pipeline.transform(rows_so_far, stores_df, features_df)
            current_week = engineered[engineered['Date'] == current_date].copy()
            X = current_week[self.feature_cols]
            current_week['Weekly_Sales'] = self.model.predict(X)

            raw_df = raw_df.merge(
                current_week[['Store', 'Dept', 'Date', 'Weekly_Sales']].rename(columns={'Weekly_Sales': '_pred'}),
                on=['Store', 'Dept', 'Date'], how='left'
            )
            raw_df['Weekly_Sales'] = raw_df['Weekly_Sales'].fillna(raw_df['_pred'])
            raw_df = raw_df.drop(columns=['_pred'])
            all_predictions.append(current_week[['Store', 'Dept', 'Date', 'Weekly_Sales']])

        return pd.concat(all_predictions, ignore_index=True)

print("Pipeline classes loaded (fixed, recursive).")

Pipeline classes loaded (fixed, recursive).


## Section 5 — Retrieve the Trained Pipeline Artifact
Pulls the already-trained pipeline from `model_experiment_XGBoost.ipynb`'s own saved run.

In [5]:
run = wandb.init(project=PROJECT, entity=ENTITY, job_type="inference-retrieve")

# Updated to retrieve the XGBoost artifact
artifact = run.use_artifact(f"{ENTITY}/{PROJECT}/xgboost_full_pipeline:latest")
artifact_dir = artifact.download()

pipeline_path = os.path.join(artifact_dir, "xgboost_full_pipeline.pkl")
with open(pipeline_path, "rb") as f:
    winning_pipeline = pickle.load(f)

print(f"Retrieved trained pipeline from artifact: {artifact.name}")
wandb.finish()

wandb:   1 of 1 files downloaded.  


Retrieved trained pipeline from artifact: xgboost_full_pipeline:latest


## Section 6 — Register to the Model Registry (Correct Current API)
`run.link_model()` -- the current method. **Wrapped defensively**: your wandb organization blocks ALL programmatic registration (confirmed with both the old `link_artifact()` and the current `link_model()` APIs -- same "migrated" error either way). This is an org-level policy, not a code bug, so the cell catches it and continues rather than crashing the notebook.

In [6]:
REGISTRY_SUCCEEDED = False
REGISTERED_MODEL_NAME = "Walmart-Sales-Best-Model"

try:
    run = wandb.init(project=PROJECT, entity=ENTITY, job_type="register-model")
    run.link_model(path=pipeline_path, registered_model_name=REGISTERED_MODEL_NAME)
    print(f"Registered '{BEST_MODEL}' pipeline as '{REGISTERED_MODEL_NAME}'")
    REGISTRY_SUCCEEDED = True
    wandb.finish()
except Exception as e:
    print("Model Registry write blocked by organization policy (confirmed, not a code issue):")
    print(f"  {e}")
    print("Continuing with the already-retrieved pipeline from Section 5 -- the trained")
    print("artifact itself is safely saved either way, just not linked into the Registry UI.")
    try:
        wandb.finish()
    except Exception:
        pass

Model Registry write blocked by organization policy (confirmed, not a code issue):
  The model registry has been migrated for teams in your organization. You may no longer make changes.
Continuing with the already-retrieved pipeline from Section 5 -- the trained
artifact itself is safely saved either way, just not linked into the Registry UI.


## Section 7 — Load the Model Directly FROM the Model Registry (if available)
Only attempts this if Section 6 actually succeeded. If the Registry write was blocked, this cell skips cleanly and Section 8 uses `winning_pipeline` from Section 5 instead -- the same trained object, retrieved via the standard artifact API rather than the Registry.

In [7]:
if REGISTRY_SUCCEEDED:
    run = wandb.init(project=PROJECT, entity=ENTITY, job_type="inference")
    registry_model_path = run.use_model(name=f"{REGISTERED_MODEL_NAME}:latest")
    print(f"Downloaded from registry: {registry_model_path}")
    with open(registry_model_path, "rb") as f:
        best_pipeline = pickle.load(f)
    print("Loaded pipeline from Model Registry.")
else:
    best_pipeline = winning_pipeline
    print("Registry unavailable -- using the pipeline already retrieved in Section 5 "
          "(same trained object, same artifact, just via the standard artifact API "
          "instead of the blocked Registry).")

Registry unavailable -- using the pipeline already retrieved in Section 5 (same trained object, same artifact, just via the standard artifact API instead of the blocked Registry).


## Section 8 — Generate Predictions on the Real Test Set

In [8]:
raw_path = '/content/drive/MyDrive/ML-final/data/raw/'
test_raw = pd.read_csv(raw_path + 'test.csv')
test_raw['Date'] = pd.to_datetime(test_raw['Date'])
stores_raw = pd.read_csv(raw_path + 'stores.csv')
features_raw = pd.read_csv(raw_path + 'features.csv')
features_raw['Date'] = pd.to_datetime(features_raw['Date'])

predictions_df = best_pipeline.predict(test_raw, stores_raw, features_raw)

submission = pd.DataFrame({
    'Id': (predictions_df['Store'].astype(str) + '_' + predictions_df['Dept'].astype(str) + '_' +
           pd.to_datetime(predictions_df['Date']).dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': predictions_df['Weekly_Sales']
})

sub_path = '/content/drive/MyDrive/ML-final/submissions/final_inference_submission.csv'
os.makedirs(os.path.dirname(sub_path), exist_ok=True)
submission.to_csv(sub_path, index=False)

print(submission.head())
print(f"\nSaved: {sub_path}")
print(f"Rows: {submission.shape[0]}")

wandb.finish()

               Id  Weekly_Sales
0  1_1_2012-11-02  33004.796875
1  1_2_2012-11-02  46270.867188
2  1_3_2012-11-02   9223.612305
3  1_4_2012-11-02  38851.441406
4  1_5_2012-11-02  29652.648438

Saved: /content/drive/MyDrive/ML-final/submissions/final_inference_submission.csv
Rows: 115064


## Section 9 — Summary

In [9]:
print(f"Best model: {BEST_MODEL}")
print(f"Internal WMAE: {results[results['Model'] == BEST_MODEL]['Internal_WMAE'].iloc[0]}")
if REGISTRY_SUCCEEDED:
    print(f"Registered as: '{REGISTERED_MODEL_NAME}' (confirmed in Model Registry)")
else:
    print(f"Model Registry write blocked by organization policy (confirmed with both old and")
    print(f"current APIs). Trained pipeline artifact was saved and retrieved successfully via")
    print(f"the standard wandb artifact API instead -- see Section 6 output for details.")
print("Predictions generated and saved in Section 8.")

Best model: XGBoost
Internal WMAE: 1442.29
Model Registry write blocked by organization policy (confirmed with both old and
current APIs). Trained pipeline artifact was saved and retrieved successfully via
the standard wandb artifact API instead -- see Section 6 output for details.
Predictions generated and saved in Section 8.
